In [1]:
# Import

import os
import random
import numpy as np
import torch
import pandas as pd
import transformers
from transformers import AutoTokenizer
from datasets import Dataset
from transformers import DataCollatorWithPadding

In [2]:
# Setup, reproducibility, device
SEED = 123

def set_all_seeds(seed: int = SEED):
    # Fix all RNG sources: high variance with ~137 training docs, repeatability matters
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # no-op if no CUDA

set_all_seeds(SEED)

# Deterministic cuDNN: slightly slower, reproducible results (negligible cost on small data)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Auto device detection: CUDA -> MPS -> CPU
def get_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        print(f"[device] CUDA active -> {torch.cuda.get_device_name(0)}")
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        dev = torch.device("mps")
        print("[device] Apple MPS active")
    else:
        dev = torch.device("cpu")
        print("[device] No GPU found -> CPU (BERT will be slow)")
    return dev

device = get_device()



[device] CUDA active -> NVIDIA GeForce RTX 5070


In [3]:
# Sanity check: 'CUDA build=None' means a CPU-only torch is installed
print(f"[versions] torch={torch.__version__} | CUDA build={torch.version.cuda}")
print(f"[seed] {SEED}")

[versions] torch=2.12.1+cu130 | CUDA build=13.0
[seed] 123


In [4]:
# Load data, select columns, integrity checks, token-length profile

CSV_PATH = "..\Dati\Processed\dataset_processed_quantile1_sentences.csv"  
MODEL_NAME = "bert-base-cased"                            # cased: casing/punct are stylistic signal
KEEP_COLS = ["article_id", "topic_id", "binary_label", "fold", "text_bert"]

# Load 
df = pd.read_csv(CSV_PATH)
df = df[KEEP_COLS].copy()
df = df.reset_index(drop=True)  # avoid __index_level_0__ leaking into HF Dataset later

# Integrity checks 
print(f"[shape] {df.shape[0]} docs, {df.shape[1]} cols")
print(f"[NaN]\n{df.isna().sum()}")
print(f"[dup article_id] {df['article_id'].duplicated().sum()}")
print(f"[folds] {sorted(df['fold'].unique())}")

# Class balance overall and per fold (must stay ~2:1)
print("\n[label balance overall]")
print(df["binary_label"].value_counts(normalize=True).round(3).to_dict())
print("\n[label balance per fold]")
print(df.groupby("fold")["binary_label"].value_counts(normalize=True).round(2).unstack())
print("\n[docs per fold]")
print(df["fold"].value_counts().sort_index().to_dict())



# Token-length profile (does max_length=512 actually bite?) 
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
# add_special_tokens=True counts [CLS]/[SEP] as in real training
tok_lens = df["text_bert"].apply(lambda t: len(tok(t, add_special_tokens=True)["input_ids"]))
df["_tok_len"] = tok_lens

print("\n[token length] describe")
print(tok_lens.describe().round(1))
for thr in (128, 256, 384, 512):
    share = (tok_lens > thr).mean()
    print(f"  > {thr} tokens: {share:.1%} of docs")

[shape] 624 docs, 5 cols
[NaN]
article_id      0
topic_id        0
binary_label    0
fold            0
text_bert       0
dtype: int64
[dup article_id] 0
[folds] [0, 1, 2, 3, 4]

[label balance overall]
{1: 0.667, 0: 0.333}

[label balance per fold]
binary_label     0     1
fold                    
0             0.33  0.67
1             0.33  0.67
2             0.33  0.67
3             0.33  0.67
4             0.33  0.67

[docs per fold]
{0: 126, 1: 126, 2: 126, 3: 123, 4: 123}


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (525 > 512). Running this sequence through the model will result in indexing errors



[token length] describe
count     624.0
mean      248.2
std       179.6
min        37.0
25%       134.5
50%       213.0
75%       325.2
max      1680.0
Name: text_bert, dtype: float64
  > 128 tokens: 76.4% of docs
  > 256 tokens: 39.9% of docs
  > 384 tokens: 15.5% of docs
  > 512 tokens: 5.4% of docs


In [19]:
# How many articles are above 512 tokens?
over = df.loc[df["_tok_len"] > 512, "_tok_len"]
print(over.describe().round(0))
print("quantili:", over.quantile([.5, .75, .9, .95]).round(0).to_dict())


count      34.0
mean      751.0
std       294.0
min       519.0
25%       557.0
50%       620.0
75%       828.0
max      1680.0
Name: _tok_len, dtype: float64
quantili: {0.5: 620.0, 0.75: 828.0, 0.9: 1204.0, 0.95: 1363.0}


In [20]:
# Are the truncated (long) docs concentrated in one class?
long_mask = df["_tok_len"] > 512
print(df.loc[long_mask, "binary_label"].value_counts().to_dict())
print("share of long docs that are class 1:",
      df.loc[long_mask, "binary_label"].mean().round(2))
print("baseline share class 1 overall:", df["binary_label"].mean().round(2))

{1: 27, 0: 7}
share of long docs that are class 1: 0.79
baseline share class 1 overall: 0.67


In [5]:
# Tokenization, dynamic-padding collator

MAX_LEN = 512  # 5% of docs exceed this; head-only truncation (bias concentrated in head)

# Build HF dataset; keep 'fold' for splitting and 'article_id' for OOF tracking
base_cols = ["text_bert", "binary_label", "fold", "article_id"]
ds_full = Dataset.from_pandas(df[base_cols].reset_index(drop=True), preserve_index=False)

def tokenize_batch(batch):
    return tok(batch["text_bert"], truncation=True, max_length=MAX_LEN)


In [6]:
ds_full = ds_full.map(tokenize_batch, batched=True)
ds_full = ds_full.rename_column("binary_label", "labels")

# Dynamic per-batch padding (not fixed 512) -> less wasted compute/memory
collator = DataCollatorWithPadding(tokenizer=tok)

# Columns the model's forward actually consumes; everything else gets stripped
MODEL_COLS = ["input_ids", "attention_mask", "token_type_ids", "labels"]

Map:   0%|          | 0/624 [00:00<?, ? examples/s]

In [7]:
# Per-fold split builder

def make_fold(k):
    """Split for fold k. Returns cleaned train/eval datasets (tensor cols only),
    plus eval article_ids and gold labels (in eval order) for OOF tracking."""
    folds = ds_full["fold"]
    train_idx = [i for i, f in enumerate(folds) if f != k]
    eval_idx  = [i for i, f in enumerate(folds) if f == k]

    train_ds = ds_full.select(train_idx)
    eval_ds  = ds_full.select(eval_idx)

    # Capture identity BEFORE dropping string columns
    eval_ids    = list(eval_ds["article_id"])
    eval_labels = list(eval_ds["labels"])

    # Strip non-model columns (a string col would break the padding collator)
    drop = [c for c in train_ds.column_names if c not in MODEL_COLS]
    train_ds = train_ds.remove_columns(drop)
    eval_ds  = eval_ds.remove_columns(drop)

    return train_ds, eval_ds, eval_ids, eval_labels

In [8]:
# Sanity check on one fold 
tr, ev, ids, y = make_fold(0)
print(f"[fold 0] train={len(tr)}  eval={len(ev)}")
print(f"[fold 0] model cols = {tr.column_names}")
print(f"[fold 0] eval class balance = "
      f"{pd.Series(y).value_counts(normalize=True).round(2).to_dict()}")

[fold 0] train=498  eval=126
[fold 0] model cols = ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
[fold 0] eval class balance = {1: 0.67, 0: 0.33}


In [11]:
#Model, weighted loss, metrics

import numpy as np
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoModelForSequenceClassification, Trainer

FREEZE_LAYERS = 0  # 0 = full fine-tuning (primary). Set e.g. 6 for the frozen ablation.

def build_model(freeze_layers: int = FREEZE_LAYERS):
    # Re-init from pretrained EVERY fold (no weight leakage across folds).
    # Reseed first so the classifier head init is identical across folds.
    torch.manual_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    # Optional: freeze embeddings + first N encoder layers (anti-overfit / cheaper)
    if freeze_layers > 0:
        for p in model.bert.embeddings.parameters():
            p.requires_grad = False
        for layer in model.bert.encoder.layer[:freeze_layers]:
            for p in layer.parameters():
                p.requires_grad = False
    return model

def fold_class_weights(train_labels, damp=0.5):
    # Softened inverse-frequency weights from this fold's train split only
    w = compute_class_weight("balanced", classes=np.array([0, 1]), y=np.array(train_labels))
    w = w ** damp          # damp<1 softens; damp=1 full, damp=0 uniform
    w = w / w.mean()       # renormalize
    return torch.tensor(w, dtype=torch.float)

class WeightedTrainer(Trainer):
    """Trainer with class-weighted CrossEntropy. Weights set per fold before training."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = (self.class_weights.to(logits.device)
                  if self.class_weights is not None else None)
        loss = nn.functional.cross_entropy(logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),  # primary metric
        "accuracy": accuracy_score(labels, preds),
        "f1_class0": f1_score(labels, preds, pos_label=0),      # minority class watch
        "f1_class1": f1_score(labels, preds, pos_label=1),
    }

# ---- Quick check: weights on fold 0's train split ----
_, _, _, _ = make_fold(0)
tr0 = ds_full.select([i for i, f in enumerate(ds_full["fold"]) if f != 0])
w0 = fold_class_weights(tr0["labels"])
print(f"[fold 0] class weights (0,1) = {w0.tolist()}")

[fold 0] class weights (0,1) = [1.1715729236602783, 0.8284271359443665]


In [ ]:

# STEP 4 — Nested CV grid search (Model 02, BERT document-level)
# Outer folds = evaluation; inner folds = hyperparameter selection.

from transformers import TrainingArguments
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

GRID = [(2e-5, 4), (2e-5, 6), (3e-5, 4), (3e-5, 6)]  # (lr, epochs)
ALL_FOLDS = [0, 1, 2, 3, 4]
TRAIN_BS, EVAL_BS = 16, 32

def grid_args(lr, epochs):
    return TrainingArguments(
        output_dir="./_tmp_bert_grid",
        learning_rate=lr, num_train_epochs=epochs,
        per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
        weight_decay=0.01, warmup_ratio=0.1, seed=SEED,
        eval_strategy="no", save_strategy="no", logging_strategy="no",
        disable_tqdm=True, report_to="none",
        fp16=torch.cuda.is_available(),
    )

def make_split(train_folds, eval_fold):
    # Build cleaned train/eval datasets from fold ids; keep eval identity for OOF
    folds = ds_full["fold"]
    tr = ds_full.select([i for i, f in enumerate(folds) if f in train_folds])
    ev = ds_full.select([i for i, f in enumerate(folds) if f == eval_fold])
    ev_ids, ev_y = list(ev["article_id"]), list(ev["labels"])
    drop = [c for c in tr.column_names if c not in MODEL_COLS]
    return tr.remove_columns(drop), ev.remove_columns(drop), ev_ids, ev_y

def train_eval(train_ds, eval_ds, eval_y, lr, epochs, damp=0.5):
    # Fresh model + fold-local damped weights; return macro-F1 and preds
    model = build_model(FREEZE_LAYERS)
    trainer = WeightedTrainer(
        model=model, args=grid_args(lr, epochs),
        train_dataset=train_ds, eval_dataset=eval_ds,
        data_collator=collator, compute_metrics=compute_metrics,
        class_weights=fold_class_weights(train_ds["labels"], damp=damp),
    )
    trainer.train()
    preds = trainer.predict(eval_ds).predictions.argmax(-1)
    f1 = f1_score(eval_y, preds, average="macro")
    del model, trainer
    torch.cuda.empty_cache()
    return f1, preds

oof_rows, outer_scores, chosen = [], [], []

for k in ALL_FOLDS:
    inner_folds = [f for f in ALL_FOLDS if f != k]

    # --- Inner CV: score each grid point on the 4 training folds ---
    grid_scores = {g: [] for g in GRID}
    for val_fold in inner_folds:
        tr_folds = [f for f in inner_folds if f != val_fold]
        tr, ev, _, ev_y = make_split(tr_folds, val_fold)
        for g in GRID:
            f1, _ = train_eval(tr, ev, ev_y, lr=g[0], epochs=g[1])
            grid_scores[g].append(f1)

    best = max(GRID, key=lambda g: np.mean(grid_scores[g]))
    chosen.append(best)

    # --- Refit on all 4 training folds with best config, predict outer fold k ---
    tr, ev, ev_ids, ev_y = make_split(inner_folds, k)
    f1, preds = train_eval(tr, ev, ev_y, lr=best[0], epochs=best[1])
    outer_scores.append(f1)
    for aid, yt, yp in zip(ev_ids, ev_y, preds):
        oof_rows.append({"article_id": aid, "fold": k, "y_true": int(yt), "y_pred": int(yp)})
    print(f"[outer {k}] best(lr,ep)={best}  f1_macro={f1:.4f}")

oof_df = pd.DataFrame(oof_rows)
print(f"\n[nested CV] f1_macro per fold = {[round(s, 4) for s in outer_scores]}")
print(f"[nested CV] f1_macro mean±std = {np.mean(outer_scores):.4f} ± {np.std(outer_scores):.4f}")
print(f"[nested CV] chosen configs   = {chosen}")

In [15]:
# Final fit on full internal dataset, save for external OOD test

FINAL_LR, FINAL_EPOCHS = 3e-5, 6
SAVE_DIR = "./bert_m02_final"

# Full-data split: no eval, train on everything
full_ds = ds_full.remove_columns([c for c in ds_full.column_names if c not in MODEL_COLS])

final = WeightedTrainer(
    model=build_model(FREEZE_LAYERS),
    args=grid_args(FINAL_LR, FINAL_EPOCHS),
    train_dataset=full_ds,
    data_collator=collator,
    class_weights=fold_class_weights(full_ds["labels"], damp=0.5),
)
final.train()

# Persist model + tokenizer for later OOD inference
final.save_model(SAVE_DIR)
tok.save_pretrained(SAVE_DIR)
print(f"[saved] {SAVE_DIR}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'train_runtime': '35.84', 'train_samples_per_second': '104.5', 'train_steps_per_second': '6.529', 'train_loss': '0.4297', 'epoch': '6'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[saved] ./bert_m02_final
